# worker-batched スループットベンチマーク(rootノイズ / search_count可変)

`notebooks/benchmark_worker_batched.ipynb`を元に、学習ループを使わず対戦だけを
繰り返し実行してスループットを比較する構成です。今回追加した2点だけを検証します。

- `SEARCH_COUNT`をBENCHMARKSの各行で個別に指定できるようにした(元notebookは全構成共通の固定値)
- `noise: True`の行だけ`SELFPLAY_ROOT_NOISE_ENABLED=1`を設定し、`tools/mcts_root_noise.py`が
  patchした`tools/batched_tournament.py`のrootノイズを有効にする(既定は無効)

`agents/16model_pretrained_noise/cluster_00`〜`15`と`workers=9 × lanes_per_worker=400`
(`TOTAL_GAMES=3600`)は、実際に101分かかった学習iterationと同じ条件です。
学習JSONは保存せず、対戦の所要時間とプロファイルだけを見ます。

`powermetrics`によるGPU計測はsudoパスワード入力が必要なため、この版では省略しています。
CPU/RAM使用率は`psutil`のみ(sudo不要)で採取します。GPU計測が必要な場合は
`benchmark_worker_batched.ipynb`の該当セルを移植してください。


In [4]:
from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd
import psutil
from IPython.display import clear_output, display


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'tools' / 'run_matches_round_robin.py').exists():
            return candidate
    raise FileNotFoundError('pokemon-tcg-agent のリポジトリルートが見つかりません。')


ROOT = find_repo_root()
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

venv_python = ROOT / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
PYTHON = venv_python if venv_python.exists() else Path(sys.executable)
RUNNER = ROOT / 'tools' / 'run_matches_round_robin.py'

print(f'ROOT={ROOT}')
print(f'PYTHON={PYTHON}')


ROOT=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent
PYTHON=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/.venv/bin/python


## ベンチマーク条件

`BENCHMARKS`の各行に`search_count`と`noise`を指定します。`workers`・`lanes_per_worker`は
実際に101分かかったiterationと同じ値に固定し、search_count/noiseだけを変えて比較します。
比較を公平にするため、全構成で同じ`TOTAL_GAMES`を使います。


In [5]:
AGENTS = {
    f'cluster_{index:02d}': f'agents/16model_pretrained_noise/cluster_{index:02d}/src/main.py'
    for index in range(16)
}

CPU_THREADS = psutil.cpu_count(logical=True) or os.cpu_count() or 1
WORKERS = max(1, CPU_THREADS - 1)
LANES_PER_WORKER = 400
TOTAL_GAMES = WORKERS * LANES_PER_WORKER

# 比較したいsearch_count/noiseの組み合わせをここに追加する。
BENCHMARKS = [
    {'search_count': 20, 'noise': True},
]

BATCH_SIZE = 256
MAX_TURNS = 100
MAX_SELECTIONS = 500
DEVICE = 'mps' if sys.platform == 'darwin' else 'cuda'
SEED = 0
ROOT_NOISE_ALPHA = 0.3
ROOT_NOISE_EPSILON = 0.25
PAIR_COUNT = len(AGENTS) * (len(AGENTS) + 1) // 2

print(
    f'エージェント数={len(AGENTS)}, 対戦カード数={PAIR_COUNT}, '
    f'論理CPU={CPU_THREADS}, workers={WORKERS}, lanes/worker={LANES_PER_WORKER}, '
    f'総試合数={TOTAL_GAMES}, device={DEVICE}, '
    f'上限={MAX_TURNS}ターン/{MAX_SELECTIONS}選択'
)


エージェント数=16, 対戦カード数=136, 論理CPU=10, workers=9, lanes/worker=400, 総試合数=3600, device=mps, 上限=100ターン/500選択


## 実行

各構成を`run_matches_round_robin.py`へ`--search-count`を変えて渡し、`noise=True`の行だけ
selfplay用のrootノイズ環境変数を設定します。既存の`benchmark_worker_batched.ipynb`と同じ
正規表現で`stdout`のprofile行をパースします。


In [6]:
NN_PATTERN = re.compile(
    r'NN: ([\d.]+)秒 / (\d+)評価 / (\d+)batch '
    r'\(平均batch=([\d.]+), 最大=(\d+)\)'
)
IPC_PATTERN = re.compile(
    r'CPU workers: (\d+), worker NN待ち合計=([\d.]+)秒, '
    r'worker NumPy梱包=([\d.]+)秒, '
    r'中央batch収集=([\d.]+)秒, IPC request=(\d+)回'
)


def require_match(pattern: re.Pattern[str], output: str, label: str) -> re.Match[str]:
    match = pattern.search(output)
    if match is None:
        raise RuntimeError(f'{label}を実行結果から取得できません。')
    return match


def average_or_nan(values: list[float]) -> float:
    return sum(values) / len(values) if values else float('nan')


def run_benchmark(config: dict) -> dict:
    search_count = int(config['search_count'])
    noise = bool(config['noise'])
    lanes = WORKERS * LANES_PER_WORKER

    progress_dir = Path(
        tempfile.mkdtemp(prefix='benchmark_progress_', dir=RESULTS_DIR)
    )

    command = [str(PYTHON), str(RUNNER)]
    for name, agent_path in AGENTS.items():
        command.extend(['--agent', f'{name}={agent_path}'])
    command.extend([
        '--backend', 'worker-batched',
        '--device', DEVICE,
        '--workers', str(WORKERS),
        '--lanes', str(lanes),
        '--total-games', str(TOTAL_GAMES),
        '--batch-size', str(BATCH_SIZE),
        '--search-count', str(search_count),
        '--max-turns', str(MAX_TURNS),
        '--max-selections', str(MAX_SELECTIONS),
        '--seed', str(SEED),
        '--quiet',
        # 進捗のN/TOTAL_GAMES表示だけが目的で、学習には使わないため終了後に削除する。
        '--training-json-dir', str(progress_dir),
        '--training-format', 'preencoded',
    ])

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    if noise:
        env['SELFPLAY_ROOT_NOISE_ENABLED'] = '1'
        env['SELFPLAY_ROOT_NOISE_ALPHA'] = str(ROOT_NOISE_ALPHA)
        env['SELFPLAY_ROOT_NOISE_EPSILON'] = str(ROOT_NOISE_EPSILON)
    else:
        for name in ('SELFPLAY_ROOT_NOISE_ENABLED', 'SELFPLAY_ROOT_NOISE_ALPHA', 'SELFPLAY_ROOT_NOISE_EPSILON'):
            env.pop(name, None)

    cpu_samples: list[float] = []
    ram_samples: list[float] = []
    psutil.cpu_percent(interval=None)

    timestamp = time.strftime('%Y%m%d_%H%M%S')
    log_prefix = f'benchmark_root_noise_sc{search_count}_noise{int(noise)}_{timestamp}'

    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    try:
        while process.poll() is None:
            cpu_samples.append(psutil.cpu_percent(interval=1.0))
            ram_samples.append(psutil.virtual_memory().percent)
            completed = sum(1 for _ in progress_dir.glob('*.pkl'))
            clear_output(wait=True)
            print(f'search_count={search_count}, noise={noise}: {completed}/{TOTAL_GAMES}')
        stdout, stderr = process.communicate()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.communicate(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.communicate()
        raise
    finally:
        shutil.rmtree(progress_dir, ignore_errors=True)
    wall = time.perf_counter() - started

    (RESULTS_DIR / f'{log_prefix}.stdout.log').write_text(stdout, encoding='utf-8')
    (RESULTS_DIR / f'{log_prefix}.stderr.log').write_text(stderr, encoding='utf-8')
    if process.returncode != 0:
        raise RuntimeError(
            f'search_count={search_count}, noise={noise} がexit={process.returncode}で失敗しました。\n'
            f'{stderr[-2000:]}'
        )

    nn = require_match(NN_PATTERN, stdout, 'NN profile')
    ipc = require_match(IPC_PATTERN, stdout, 'IPC profile')
    nn_evaluations = int(nn.group(2))
    ipc_requests = int(ipc.group(5))

    return {
        'search_count': search_count,
        'noise': noise,
        'workers': WORKERS,
        'lanes': lanes,
        'games': TOTAL_GAMES,
        'wall_seconds': wall,
        'wall_minutes': wall / 60,
        'games/s': TOTAL_GAMES / wall,
        '平均batch': float(nn.group(4)),
        'NN評価数': nn_evaluations,
        'IPC/NN評価': ipc_requests / nn_evaluations,
        'CPU平均%': average_or_nan(cpu_samples),
        'CPU最大%': max(cpu_samples, default=float('nan')),
        'RAM平均%': average_or_nan(ram_samples),
        'stdout_log': str(RESULTS_DIR / f'{log_prefix}.stdout.log'),
    }


rows = []
for config in BENCHMARKS:
    print(f'--- running search_count={config["search_count"]}, noise={config["noise"]} ---')
    row = run_benchmark(config)
    rows.append(row)
    display(pd.DataFrame(rows))

results_frame = pd.DataFrame(rows)
results_frame


search_count=20, noise=True: 3600/3600


,search_count,noise,workers,lanes,games,wall_seconds,wall_minutes,games/s,平均batch,NN評価数,IPC/NN評価,CPU平均%,CPU最大%,RAM平均%,stdout_log
0,20,True,9,3600,3600,646.90645,10.781774,5.564947,27.5,12192834,0.006935,60.944237,97.9,79.156542,/Users/naoki/Desktop/develop/pokemon_tgc_agent...


,search_count,noise,workers,lanes,games,wall_seconds,wall_minutes,games/s,平均batch,NN評価数,IPC/NN評価,CPU平均%,CPU最大%,RAM平均%,stdout_log
0,20,True,9,3600,3600,646.90645,10.781774,5.564947,27.5,12192834,0.006935,60.944237,97.9,79.156542,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
